<a href="https://colab.research.google.com/github/back1992/ai_math/blob/main/ai_math.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get install -y poppler-utils
!pip install pdf2image

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 1s (229 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...


In [4]:
import os
from pdf2image import convert_from_path
import json
import base64
from google.colab import drive

# 1. 挂载 Google Drive (这样你可以直接读取 PDF 并保存结果)
drive.mount('/content/drive')

def prepare_tuning_data(pdf_path, output_folder, start_page=1, end_page=5):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # 将 PDF 转换为图片列表
    pages = convert_from_path(pdf_path, first_page=start_page, last_page=end_page)

    dataset = []

    for i, page in enumerate(pages):
        page_num = start_page + i
        img_name = f"page_{page_num}.jpg"
        img_path = os.path.join(output_folder, img_name)

        # 保存图片
        page.save(img_path, 'JPEG')

        # 构造微调 JSON 条目 (这里需要你提供一部分“完美答案”作为训练目标)
        # 注意：在 AI Studio 批量上传时，通常是图片文件+对应的 JSON 描述
        data_entry = {
            "instruction": "Extract all math questions from this page and format as JSON.",
            "context": "",
            "response": "这里填入你之前在 Firestore 存的该页对应完美 JSON"
        }
        dataset.append(data_entry)
        print(f"✅ 处理完成第 {page_num} 页")

    # 保存为 JSONL 格式
    with open(f'{output_folder}/training_data.jsonl', 'w', encoding='utf-8') as f:
        for entry in dataset:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')

# 使用示例
prepare_tuning_data(
    pdf_path='/content/drive/MyDrive/project/ai_math/finetune_mode.pdf',
    output_folder='/content/drive/MyDrive/project/ai_math/AList_Tuning_Project'
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 处理完成第 1 页
✅ 处理完成第 2 页
✅ 处理完成第 3 页
✅ 处理完成第 4 页
✅ 处理完成第 5 页


In [6]:
import firebase_admin
from firebase_admin import credentials, firestore
import json
import os

# 1. 初始化 Firebase (请确保你已上传 serviceAccountKey.json)
if not firebase_admin._apps:
    cred = credentials.Certificate('/content/drive/MyDrive/project/ai_math/finetune_model/serviceAccountKey.json')
    firebase_admin.initialize_app(cred)

db = firestore.client()

def generate_final_training_jsonl(output_file, pdf_total_pages=90):
    with open(output_file, 'w', encoding='utf-8') as f:
        for page_num in range(1, pdf_total_pages + 1):
            # 获取对应的 Firestore 数据 (假设你的 Doc ID 是 page_1, page_2...)
            doc_ref = db.collection('extracted_questions').document(f'page_{page_num}')
            doc = doc_ref.get()

            # 定义基础结构
            entry = {
                "instruction": "Extract all math questions from this page and format as JSON.",
                "context": f"This is page {page_num} of the book.",
                "response": ""
            }

            # 逻辑判定
            if page_num <= 5:
                # 针对前 5 页的过滤训练
                entry["response"] = json.dumps({
                    "is_question_page": False,
                    "reason": "Introduction/Cover page",
                    "questions": []
                }, ensure_ascii=False)
            elif doc.exists:
                # 针对有题目的页面，填入真实的完美 JSON
                real_data = doc.to_dict()
                entry["response"] = json.dumps(real_data, ensure_ascii=False)
            else:
                # 如果这页还没校对，先跳过或设为待处理
                print(f"⚠️ 第 {page_num} 页在 Firestore 中未找到，跳过。")
                continue

            # 写入一行
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
            print(f"✅ 已对齐第 {page_num} 页数据")

# 执行
generate_final_training_jsonl('/content/drive/MyDrive/project/ai_math/finetune_model/final_training_data.jsonl')

✅ 已对齐第 1 页数据
✅ 已对齐第 2 页数据
✅ 已对齐第 3 页数据
✅ 已对齐第 4 页数据
✅ 已对齐第 5 页数据
⚠️ 第 6 页在 Firestore 中未找到，跳过。
⚠️ 第 7 页在 Firestore 中未找到，跳过。
⚠️ 第 8 页在 Firestore 中未找到，跳过。
⚠️ 第 9 页在 Firestore 中未找到，跳过。
⚠️ 第 10 页在 Firestore 中未找到，跳过。
⚠️ 第 11 页在 Firestore 中未找到，跳过。
⚠️ 第 12 页在 Firestore 中未找到，跳过。
⚠️ 第 13 页在 Firestore 中未找到，跳过。
⚠️ 第 14 页在 Firestore 中未找到，跳过。
⚠️ 第 15 页在 Firestore 中未找到，跳过。
⚠️ 第 16 页在 Firestore 中未找到，跳过。
⚠️ 第 17 页在 Firestore 中未找到，跳过。
⚠️ 第 18 页在 Firestore 中未找到，跳过。
⚠️ 第 19 页在 Firestore 中未找到，跳过。
⚠️ 第 20 页在 Firestore 中未找到，跳过。
⚠️ 第 21 页在 Firestore 中未找到，跳过。
⚠️ 第 22 页在 Firestore 中未找到，跳过。
⚠️ 第 23 页在 Firestore 中未找到，跳过。
⚠️ 第 24 页在 Firestore 中未找到，跳过。
⚠️ 第 25 页在 Firestore 中未找到，跳过。
⚠️ 第 26 页在 Firestore 中未找到，跳过。
⚠️ 第 27 页在 Firestore 中未找到，跳过。
⚠️ 第 28 页在 Firestore 中未找到，跳过。
⚠️ 第 29 页在 Firestore 中未找到，跳过。
⚠️ 第 30 页在 Firestore 中未找到，跳过。
⚠️ 第 31 页在 Firestore 中未找到，跳过。
⚠️ 第 32 页在 Firestore 中未找到，跳过。
⚠️ 第 33 页在 Firestore 中未找到，跳过。
⚠️ 第 34 页在 Firestore 中未找到，跳过。
⚠️ 第 35 页在 Firestore 中未找到，跳过。
⚠️ 第 36 页在 Firestore 中未找到，跳过。
⚠️ 第 37 页